# Семинар 6. Эмбеддинги слов и классификация текстов

На занятии мы попробуем на практике то, что изучили на лекции - посмотрим на операции с эмбеддингами, попробуем самостоятельно обучить word2vec, а также решить с их помощью прикладную задачу классификации текстов. Кроме того, посмотрим на нейросетевой подход к решению данной задачи, а именно, как использовать MLP для обработки текстов.

Для загрузки предобученных эмбеддингов и датасета IMDb нужен доступ к сети. Обучающие ячейки можно запускать на GPU; на CPU уменьшите число эпох.

## Источники

- [Конспект по основам NLP](https://github.com/mmp-practicum-team/mmp_practicum_spring_2024/blob/main/Seminars/Seminar%2007/%D0%AF%D0%B7%D1%8B%D0%BA%D0%BE%D0%B2%D1%8B%D0%B5%20%D0%BC%D0%BE%D0%B4%D0%B5%D0%BB%D0%B8.%20Torchtext.ipynb)

- [Документация gensim](https://radimrehurek.com/gensim/models/word2vec.html)

- **UMAP**: [Leland McInnes et al., 2018](https://arxiv.org/abs/1802.03426), примерно 22 тысячи цитирований

- [Документация umap](https://umap-learn.readthedocs.io/en/latest/)

- [Выделение смысловых компонент слов с помощью Lasso](https://lars76.github.io/2021/06/16/operations-contextual-embeddings.html)

- [Датасет рецензий с Imdb](https://ai.stanford.edu/~amaas/data/sentiment/).

- [Документация CatBoost](https://catboost.ai/docs/en/concepts/python-reference_catboostclassifier)

- [Примеры датасетов для классификации текстов](https://lena-voita.github.io/nlp_course/text_classification.html#dataset_examples)

- [Документация fasttext](https://fasttext.cc/docs/en/python-module.html)

- [Документация tokenizers](https://huggingface.co/docs/tokenizers/index)

- [Документация datasets](https://huggingface.co/docs/datasets/index)

- Для большинства изображений указан источник, откуда оно взято, как правило, являющийся статьей

## Подготовка окружения

In [ ]:
!pip install gensim
!pip install catboost
!pip install tokenizers


In [ ]:
import os
import numpy as np
import random

import matplotlib_inline
import matplotlib.pyplot as plt

%matplotlib inline


np.random.seed(52)
random.seed(52)

## Модели эмбеддингов на основе предсказаний

Мы будем пользоваться библиотекой [gensim](https://radimrehurek.com/gensim/models/word2vec.html), так как в ней удобно реализованы различные методы обучения эмбеддингов: `word2vec`.

Кроме того, существуют уже предобученные модели, что позволяет в несколько строк получить качественные эмбеддинги. Посмотрим на список доступных моделей ниже.

In [ ]:
import gensim
import gensim.downloader

In [ ]:
print(list(gensim.downloader.info()['models'].keys()))

Для наглядности мы будем использовать `word2vec-ruscorpora-300`, так как это обученные эмбеддинги для русского языка.

In [ ]:
word2vec_rus = gensim.downloader.load('word2vec-ruscorpora-300')

In [ ]:
print(f"Всего слов {len(word2vec_rus.index_to_key)}")
print(f"Размерность {word2vec_rus.vector_size}")

print(f"Примеры слов:", ', '.join(word2vec_rus.index_to_key[:10]))


Можно заметить, что каждое слово имеет суффикс, который отвечает за часть речи (NOUN - существительное, VERB - глагол и тд). Задача определения части речи называется `part-of-speech tagging`.

Красивую визуализацию можно найти [тут](https://rusvectores.org/ru/visual/).

Мы же попробуем визуализировать и изучить интересные свойства самостоятельно.

## Операции над эмбеддингами

В моделях из библиотеки `gensim` реализован автоматический поиск наиболее похожих слов, то есть поиск слов, чьи эмбеддинги наиболее близки к эмбеддингу исходного.

In [ ]:
a = word2vec_rus.get_vector('холод_NOUN')

word2vec_rus.similar_by_vector(a)

Кроме того, мы можем производить над эмбеддингами арифметические операции.

In [ ]:
a = word2vec_rus.get_vector('король_NOUN')
b = word2vec_rus.get_vector('мужчина_NOUN')

word2vec_rus.similar_by_vector(a - b)

In [ ]:
c = word2vec_rus.get_vector('женщина_NOUN')

word2vec_rus.similar_by_vector(a - b + c)

In [ ]:
a = word2vec_rus.get_vector('москва_NOUN')
b = word2vec_rus.get_vector('россия_NOUN')
c = word2vec_rus.get_vector('германия_NOUN')

word2vec_rus.similar_by_vector(a - b + c)

Чуть аккуратней этот функционал уже реализован в библиотеке.

In [ ]:
word2vec_rus.most_similar(
    positive=['москва_NOUN', 'германия_NOUN'],
    negative=['россия_NOUN']
)

У некоторых слов много значений, поэтому мы можем найти достаточно разные слова по смыслу.

In [ ]:
word2vec_rus.similar_by_vector('ручка_NOUN')

Кроме того, можно быстро находить синонимы.

In [ ]:
word2vec_rus.similar_by_vector('красивый_ADJ')

Мы ожидаем, что для разных по смыслу слов расстояние между эмбеддингами будет больше, чем между похожими.

In [ ]:
word2vec_rus.distance('щука_NOUN', 'тумбочка_NOUN')

In [ ]:
word2vec_rus.distance('щука_NOUN', 'окунь_NOUN')

In [ ]:
word2vec_rus.distance('щука_NOUN', 'карась_NOUN')

In [ ]:
word2vec_rus.distance('щука_NOUN', 'рыба_NOUN')

Однако не стоит забывать, что эмбеддинги не идеальные, поэтому иногда возникают странные ситуации.

In [ ]:
word2vec_rus.distance('щука_NOUN', 'вода_NOUN')

В случае более хороших эмбеддингов, полученных из нейронных сетей, можно даже обучать линейные модели для поиска "смысловых компонент" слова. Подробнее можно изучить [по ссылке](https://lars76.github.io/2021/06/16/operations-contextual-embeddings.html).

## Классификация

Мы хотим использовать векторные представления слов для алгоритмов машинного обучения или нейронных сетей. Прежде чем перейти к нейронным сетям, нам нужны бейзлайны - то есть то, с чем будем сравниваться. Сначала проверим, как работают стандартные алгоритмы машинного обучения на эмбеддингах.

Мы будем решать задачу бинарной классификации для [датасета рецензий с Imdb](https://ai.stanford.edu/~amaas/data/sentiment/).

В качестве базовой модели будем использовать [CatBoost](https://catboost.ai/docs/en/concepts/python-reference_catboostclassifier).

Подробный список классических датасетов для классификации текста можно посмотреть [здесь](https://lena-voita.github.io/nlp_course/text_classification.html#dataset_examples).

In [ ]:
if not os.path.exists("aclImdb"):
    if not os.path.exists("aclImdb_v1.tar.gz"):
        ! wget https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
    ! tar -xf aclImdb_v1.tar.gz


Мы не будем усердно заниматься предобработкой, так как наша цель - именно сравнить методы, а не добиться более высокого качества. В качестве предобработки мы приведем все символы к нижнему регистру.

In [ ]:
def preprocess(l):
    return l.lower()

In [ ]:
def get_dataset(split='train', base_dir='aclImdb'):
    path_data   = os.path.join(base_dir, split)
    path_data_n = os.path.join(path_data, 'neg')
    path_data_p = os.path.join(path_data, 'pos')
    path_neg  = list(os.listdir(path_data_n))
    path_neg  = sorted(path_neg)
    path_pos  = list(os.listdir(path_data_p))
    path_pos  = sorted(path_pos)


    print(f"Объектов отрицательного класса {len(path_neg)}")
    print(f"Объектов положительного класса {len(path_pos)}")

    X = []
    y = [0] * len(path_neg) + [1] * len(path_pos)

    for p in path_neg:
        with open(os.path.join(path_data_n, p), 'r', encoding='utf-8') as f:
            l = f.read()
            X.append(preprocess(l))

    for p in path_pos:
        with open(os.path.join(path_data_p, p), 'r', encoding='utf-8') as f:
            l = f.read()
            X.append(preprocess(l))

    return X, y

In [ ]:
X_train, y_train = get_dataset(split='train')

In [ ]:
X_test, y_test = get_dataset(split='test')

Посмотрим на примеры предложений.

In [ ]:
X_train[0]

In [ ]:
X_test[0]

Для обучения моделей эмбеддингов, нам нужно разделить слова на структурные единицы (символы, слова, n-граммы). Мы будем работать на уровне слов.

In [ ]:
def get_corpus(X_data):

    return list(x.split() for x in X_data)

In [ ]:
corpus_train = get_corpus(X_train)
corpus_test  = get_corpus(X_test)

In [ ]:
corpus_train[0][:10]

## word2vec

Обучим word2vec модель на наших данных.


In [ ]:
main_corpus = corpus_train

In [ ]:
%%time
model_wv = gensim.models.Word2Vec(sentences=main_corpus, vector_size=100, window=5, min_count=1, workers=1, seed=52)

Эмбеддинг слова является вектором размерности 100, при этом каждое предложение имеет разную длину. Мы (пока) не умеем обучать модель на данных произвольного размера, поэтому нам нужно как-то агрегировать эмбеддинги по всему предложению.

Мы будем использовать средний эмбеддинг предложения, как фичу для обучения классификатора. В таком подходе мы **не учитываем порядок слов**, то есть теряем много информации.

In [ ]:
def encode_corpus(corpus, encode_fn):

    corp_enc = []
    for sentences in corpus:
        sent_enc = list(encode_fn(w) for w in sentences)
        sent_enc = np.array(sent_enc).mean(axis=0)

        corp_enc.append(sent_enc)

    return np.array(corp_enc)

In [ ]:
enc_train_wv = encode_corpus(
    [[word for word in sentence if word in model_wv.wv] for sentence in corpus_train],
    lambda word: model_wv.wv[word]
)
enc_test_wv = encode_corpus(
    [[word for word in sentence if word in model_wv.wv] for sentence in corpus_test],
    lambda word: model_wv.wv[word]
)


In [ ]:
from catboost import CatBoostClassifier

In [ ]:
def accuracy(y_pred, y_test):
    acc = np.mean(y_pred == y_test)
    print(f"Точность {acc:0.3f}")

In [ ]:
%%time
clf = CatBoostClassifier(random_seed=52)
clf.fit(enc_train_wv, y_train, verbose=False)

In [ ]:
y_pred = clf.predict(enc_test_wv)
accuracy(y_pred, y_test)

## Нейронный подход

Мы рассмотрим, как токенизировать тексты с помощью библиотеки `tokenizers`, чтобы работа с токенами происходила независимо от архитектуры и обучения.

Важной частью будет сравнение результатов с подходами из классического машинного обучения.


In [ ]:
from tqdm import tqdm
from functools import partial


import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

import tokenizers
from tokenizers import Tokenizer, trainers

os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
def set_global_seed(seed: int) -> None:
    """Set global seed for reproducibility.
    :param int seed: Seed to be set
    """
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def print_params_count(model):

    total_params = sum(p.numel() for p in model.parameters())
    total_params_grad = sum(p.numel() for p in model.parameters() if p.requires_grad)

    model_name = model.__class__.__name__
    print(f"Информация о числе параметров модели: {model_name}")
    print(f"Всего параметров: \t\t {total_params}")
    print(f"Всего обучаемых параметров: \t {total_params_grad}")
    print()

In [ ]:
set_global_seed(52)

### tokenizers

Прежде чем создавать модель, нам необходимо создать словарь и произвести токенизацию предложений. Именно от словаря будет зависеть, насколько много "слов" знает наша модель. Мы воспользуемся библиотекой [tokenizers](https://huggingface.co/docs/tokenizers/index). Библиотека `tokenizers` является частью инфраструктуры `Hugging Face`, с которой вы познакомитесь на ближайших парах.


Токенизация происходит через класс `tokenizer`. Для того чтобы получить `tokenizer` его надо сначала **обучить**, для этого нам необходимо использовать `tokenizers.trainers`. Так как мы будем работать на уровне слов, то выберем `tokenizers.trainers.WordLevelTrainer`.

Для работы с текстами нам необходимо зарезервировать два специальных токена:
1. `<pad>` для токена означающего паддинг
2. `<unk>` для токенов, которые отсутствуют в словаре

#### `WordLevelTrainer`

Будем рассматривать словарь размером 5000 слов.


```python
trainers.WordLevelTrainer(self, /, *args, **kwargs)
Docstring:     
Trainer capable of training a WorldLevel model

Args:
    vocab_size (:obj:`int`, `optional`):
        The size of the final vocabulary, including all tokens and alphabet.

    min_frequency (:obj:`int`, `optional`):
        The minimum frequency a pair should have in order to be merged.

    show_progress (:obj:`bool`, `optional`):
        Whether to show progress bars while training.

    special_tokens (:obj:`List[Union[str, AddedToken]]`):
        A list of special tokens the model should know of.
```

In [ ]:
top_n_words = 5_000

In [ ]:
trainer = trainers.WordLevelTrainer(
    vocab_size     = top_n_words,
    special_tokens = ["<pad>", "<unk>"],
    show_progress  = True
)

tokenizer = tokenizers.Tokenizer(
    model=tokenizers.models.WordLevel()
)

Мы будем обучаться из памяти, поэтому нам необходимо передать итератор, который пройдет по всем текстам, которые уже разбиты на слова.

In [ ]:
text_data = corpus_train
text_data[0][:10]

In [ ]:
tokenizer.train_from_iterator(tqdm(text_data, total=len(text_data)), trainer=trainer)

Посмотрим на словарь.

In [ ]:
print(f"Размер словаря {tokenizer.get_vocab_size()}")

print(list(tokenizer.get_vocab().keys())[:10])

for idx in [0, 1, 2, 21, 42]:
    print(f"idx = {idx}, word = {tokenizer.id_to_token(idx)}")

**Важно:** Cпециальные токены имеют наименьшие *id* по умолчанию.

Для кодирования предложений используется метод `encode`, установим `is_pretokenized=True`.


In [ ]:
tokenized_text = text_data[0][:20]

print("Исходный текст:")
print(tokenized_text)

result = tokenizer.encode(tokenized_text, is_pretokenized=True)
print(result)

print("Tokens ids: ", result.ids)

Для декодирования воспользуемся методом `decode`, мы хотим посмотреть насколько словарь из 5 тысяч слов выразительный, то есть какие слова мы пропускаем и понятен ли смысл предложения без них.

In [ ]:
print("Исходный текст:")
print(tokenized_text)

print("Результат декодирования:")
print(tokenizer.decode(result.ids, skip_special_tokens=False))

В примере выше мы видим, что в нашем словаре нет слова `unnatural` и слова `pig.`. Возможно, если бы мы лучше предобработали наш текст, то таких проблем не возникло бы.

### Данные

#### Датасет

Часто бывает полезно ограничить длину предложений параметром `max_len`, чтобы подавать в модель не слишком длинные тексты.


In [ ]:
class TextDataset(Dataset):
    def __init__(self, texts, targets, tokenizer, max_len=20):
        super().__init__()

        self.max_len   = max_len
        self.texts     = texts
        self.targets   = targets
        self.tokenizer = tokenizer
        self.tokens    = []

        for text in tqdm(texts):
            tokens = self.tokenizer.encode(text, is_pretokenized=True).ids
            self.tokens.append(tokens)

    def __getitem__(self, idx):
        """
        :param int idx: index of object in dataset
        :return dict: Dictionary with all useful object data
            {
                'text' str: unprocessed text,
                'target' torch.Tensor(dtype=torch.long): sentiment of the text (0 for negative, 1 for positive)
                'tokens' torch.Tensor(dtype=torch.long): tensor of tokens ids for the text
                'tokens_len' torch.Tensor(dtype=torch.long): number of tokens
            }
        """

        tokens = self.tokens[idx]
        tokens = tokens[:self.max_len]

        return {
            'text': self.texts[idx],
            'target': torch.tensor(self.targets[idx], dtype=torch.long),
            'tokens': torch.tensor(tokens, dtype=torch.long),
            'tokens_len': torch.tensor(len(tokens), dtype=torch.long),
        }

    def __len__(self):
        """
        :return int: number of objects in dataset
        """
        return len(self.targets)

In [ ]:
train_dataset = TextDataset(corpus_train, y_train, tokenizer)

Посмотрим на данные.

In [ ]:
train_dataset[0]

In [ ]:
train_dataset[-1]

#### Даталоадер

Даталоадер предназначен для того, чтобы эффективно подавать данные из датасета в нашу модель. Он разбивает их на батчи и перемешивает их, а также реализует функционал параллельной подгрузки батчей на device. Чтобы объединить несколько тензоров разной длины в один можно использовать функцию `torch.nn.utils.rnn.pad_sequence`

Обратите внимание на ее аргументы:
1. `batch_first` определяет по какой оси "складывать" тензоры. Мы будем использовать `batch_first=True`, так как это удобнее для восприятия.
2. `padding_value` - число, которое будет использоваться в качестве паддинга, чтобы сделать все тензоры одинаковой длины

In [ ]:
torch.nn.utils.rnn.pad_sequence([
    torch.tensor([1, 2, 3]),
    torch.tensor([4, 5]),
    torch.tensor([6, 7, 8, 9])
], batch_first=True, padding_value=-1)

In [ ]:
def collate_fn(batch, padding_value, batch_first=True):
    """
    :param List[Dict] batch: List of objects from dataset
    :param int padding_value: Value that will be used to pad tokens
    :param bool batch_first: If True resulting tensor with tokens must have shape [B, T] otherwise [T, B]
    :return dict: Dictionary with all data collated
        {
            'targets' torch.Tensor(dtype=torch.long): sentiment of the text for each object in batch
            'texts' List[str]: All texts in one list
            'tokens' torch.Tensor(dtype=torch.long): tensor of tokens ids padded with @padding_value
            'tokens_lens' torch.Tensor(dtype=torch.long): number of tokens for each object in batch
        }
    """

    texts       = [obj['text'] for obj in batch]
    targets     = torch.stack([obj['target'] for obj in batch])
    tokens_lens = torch.stack([obj['tokens_len'] for obj in batch])
    tokens      = torch.nn.utils.rnn.pad_sequence(
        [obj['tokens'] for obj in batch], batch_first=batch_first, padding_value=padding_value
    )

    return {
        'targets': targets,

        'texts': texts,
        'tokens': tokens,
        'tokens_lens': tokens_lens
    }

In [ ]:
train_dataloader = DataLoader(
    train_dataset, batch_size=4, shuffle=True, num_workers=0,
    collate_fn=partial(collate_fn, padding_value=tokenizer.token_to_id('<pad>'))
)

Посмотрим на какой-нибудь батч:

In [ ]:
batch = next(iter(train_dataloader))
batch.keys(), batch['targets'], batch['tokens'], batch['tokens_lens']

### Эмбеддинги в PyTorch

При обучении нейронной сети мы будем сами обучать эмбеддинги токенов. Для этого нам нужно использовать класс `torch.nn.Embedding`, который по сути реализует массив.

Эмбеддинг токена $i$ это строка $i$ в матрице эмбеддингов.

- `num_embeddings` - число токенов

- `embedding_dim` - размерность эмбеддинга

- `padding_idx` - индекс токена `<pad>`

```python
torch.nn.Embedding(
    num_embeddings: int,
    embedding_dim: int,
    padding_idx: Optional[int] = None,
    max_norm: Optional[float] = None,
    norm_type: float = 2.0,
    scale_grad_by_freq: bool = False,
    sparse: bool = False,
    _weight: Optional[torch.Tensor] = None,
    _freeze: bool = False,
    device=None,
    dtype=None,
) -> None
```

Посмотрим как преобразуется батч после слоя `nn.Embedding`.

In [ ]:
embed_dim = 128

emb_layer = torch.nn.Embedding(
    tokenizer.get_vocab_size(), embed_dim,
    padding_idx=tokenizer.token_to_id('<pad>')
)

In [ ]:
out = emb_layer(batch['tokens'])

out.shape

Так как `nn.Embedding` реализует матрицу, то число параметров зависит от размера словаря и размерности эмбеддингов.

In [ ]:
tokenizer.get_vocab_size() * embed_dim

In [ ]:
print_params_count(emb_layer)

### Полносвязная нейронная сеть (MLP)

Выше мы уже создали удобные классы-обертки для работы с данными. Теперь мы соберем модель для решения задачи классификации.


In [ ]:
class MLPClassifier(torch.nn.Module):
    def __init__(
        self, embedding_dim, hidden_dim, output_size, tokenizer: Tokenizer,
        dropout=0.5
    ):
        super().__init__()

        self.dropout = dropout

        self.tokenizer = tokenizer
        self.hidden_dim = hidden_dim
        self.output_size = output_size
        self.embedding_dim = embedding_dim

        # Create a simple lookup table that stores embeddings of a fixed dictionary and size.
        #    Use torch.nn.Embedding.
        self.word_embeddings = torch.nn.Embedding(
            self.tokenizer.get_vocab_size(), self.embedding_dim,
            padding_idx=self.tokenizer.token_to_id('<pad>')
        )

        self.features = torch.nn.Sequential(
            nn.Linear(self.embedding_dim, self.hidden_dim),
            nn.ReLU(), nn.Dropout(self.dropout),
            nn.Linear(self.hidden_dim,    self.hidden_dim),
            nn.ReLU(), nn.Dropout(self.dropout),
            nn.Linear(self.hidden_dim, self.hidden_dim),
            nn.ReLU(), nn.Dropout(self.dropout),
        )
        # Create linear layer for classification
        self.output = torch.nn.Sequential(
            torch.nn.Linear(self.hidden_dim, self.output_size),
        )

    def forward(self, tokens):
        """
        :param torch.Tensor(dtype=torch.long) tokens: Batch of texts represented with tokens.
        :return torch.Tensor(dtype=torch.long): Vector representation for each sequence in batch
        """
        # Evaluate embeddings
        embedding = self.word_embeddings(tokens)

        # Make forward pass through MLP network
        mask = (tokens != self.tokenizer.token_to_id('<pad>')).unsqueeze(-1)
        h = (embedding * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1)
        h   = self.features(h)
        out = self.output(h)

        return out

In [ ]:
net = MLPClassifier(
    embedding_dim = 128,
    hidden_dim    = 64,
    output_size   = 2,
    tokenizer     = tokenizer
)

out = net(batch['tokens'])
out.shape

In [ ]:
print_params_count(net)